# Imports

In [ ]:
import sys
import os
import pickle

sys.path.append(os.path.abspath('../src'))
from D_Compute_Spectrograms.load_interpolation_config import load_interpolation_config
from E_Plot_Spectrograms.load_average_tfr_results import load_average_tfr_results
from E_Plot_Spectrograms.plot_average_tfr import plot_average_tfr
from E_Plot_Spectrograms.load_matlab_colormap import load_matlab_colormap
from E_Plot_Spectrograms.create_grouped_interpolation_config import create_grouped_interpolation_config
from E_Plot_Spectrograms.plot_spectrograms_on_capmap import plot_spectrograms_on_capmap
from F_Cluster_perm_test.resolve_montage_to_DigMontage import resolve_montage_to_DigMontage
from B_EEG_Preprocessing.set_montage_and_check import set_montage_and_check
from C_EEG_Epoching.load_clean_eeg import load_clean_eeg

# Variables

In [ ]:
subjectID = 'Pilote002'
date      = '2026-07-10'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
start_time = -0.7
end_baseline = 0.0
end_time = 2.7
fmin = 4
fmax = 40

### Load Data

In [ ]:
if task=='Task1_PPS':
    folder_name = "TFR_interp_raw"
else:
    folder_name = "TFR_raw"

file_path_raw = os.path.join(data_path, subjectID, task, "Output", folder_name, f"{subjectID}_{date}_{task}_raw_tfr_average")
tfr_average_dict_raw = load_average_tfr_results(file_path_raw)

if task=='Task1_PPS':
    config_path_raw = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_per_trial_type.json')
    interpolation_config_raw = load_interpolation_config(config_path_raw)

    file_path_grouped = os.path.join(data_path, subjectID, task, "Output", "TFR_grouped", f"{subjectID}_{date}_{task}_grouped_tfr_average")
    tfr_single_dict_grouped = load_average_tfr_results(file_path_grouped)

    config_path_merged = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_for_merging.json')
    interpolation_config_merged = load_interpolation_config(config_path_merged)

    with open(os.path.join(os.path.dirname(file_path_grouped), "groups_dict.pkl"), "rb") as f:
        groups = pickle.load(f)

colormap_path = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'E_Plot_Spectrograms', 'NRcolormap.mat')
NRcmap = load_matlab_colormap(colormap_path, "NRcolormap")

### Plot Non-merged Spectrograms

In [ ]:
if task=='Task1_PPS':
    interpolation_config=interpolation_config_raw
else:
    interpolation_config=None


output_path_raw = os.path.join(data_path, subjectID, task, 'Output', 'Spectrograms_raw')

for event_name in tfr_average_dict_raw.keys():
    plot_average_tfr(
        tfr_average_dict=tfr_average_dict_raw,
        event_name=event_name,
        interpolation_config=interpolation_config_raw,
        t_min=start_time,
        t_max=end_time,
        f_min=fmin,
        f_max=fmax,
        output_path=output_path_raw,
        colormin=-5,
        colormax=5,
        cmap=NRcmap
    )

### Plot merged Spectrograms

In [ ]:
if task=='Task1_PPS':
    # Create a new timimg dict of the events timimg for the grouped data (for plot use)
    grouped_interpolation_config = create_grouped_interpolation_config(groups,interpolation_config_merged)

    output_path_merged = os.path.join(data_path, subjectID, task, 'Output', 'Spectrograms_merged')
    # Plot the averaged tfr
    for event_name in tfr_single_dict_grouped.keys():
        plot_average_tfr(
            tfr_average_dict=tfr_single_dict_grouped,
            event_name=event_name,
            interpolation_config=grouped_interpolation_config,
            t_min=start_time,
            t_max=end_time,
            f_min=fmin,
            f_max=fmax,
            output_path=output_path_merged,
            cmap=NRcmap,
            groups=groups
        )

### Plot merged Spectrograms on CapMap

In [ ]:
if task=='Task1_PPS':
    # Create a new timimg dict of the events timimg for the grouped data (for plot use)
    grouped_interpolation_config = create_grouped_interpolation_config(groups,interpolation_config_merged)

    raw_cleaned, _, _ = load_clean_eeg(data_path, subjectID, date, task)
    raw_montage = set_montage_and_check(raw_cleaned, plot=False)

    # Plot the averaged tfr
    for event_name, tfr in tfr_single_dict_grouped.items():
        montage = resolve_montage_to_DigMontage(raw_montage)
        tfr.info.set_montage(montage,on_missing="warn")       
        plot_spectrograms_on_capmap(
            tfr=tfr,
            subjectID=subjectID,
            task=task,
            trial_type=event_name,
            n_epochs=str(tfr.nave),
            t_min=start_time,
            t_max=end_time,
            f_min=fmin,
            f_max=fmax,
            data_path=data_path,
            interpolation_config=grouped_interpolation_config,
            groups=groups,
            cmap=NRcmap
        )


    for event_name, tfr in tfr_average_dict_raw.items():
        if "VisualOnly" in event_name:
            montage = resolve_montage_to_DigMontage(raw_montage)

            tfr.info.set_montage(
                montage,
                on_missing="warn"
            )

            plot_spectrograms_on_capmap(
            tfr=tfr,
            subjectID=subjectID,
            task=task,
            trial_type=event_name,
            n_epochs=str(tfr.nave),
            t_min=start_time,
            t_max=end_time,
            f_min=fmin,
            f_max=fmax,
            data_path=data_path,
            interpolation_config=interpolation_config_raw,
            groups=groups,
            cmap=NRcmap
        )
            
else:
    raw_cleaned, _, _ = load_clean_eeg(data_path, subjectID, date, task)
    raw_montage = set_montage_and_check(raw_cleaned, plot=False)

    for event_name, tfr in tfr_average_dict_raw.items():
        montage = resolve_montage_to_DigMontage(raw_montage)
        tfr.info.set_montage(montage,on_missing="warn")  

        plot_spectrograms_on_capmap(
                    tfr=tfr,
                    subjectID=subjectID,
                    task=task,
                    trial_type=event_name,
                    n_epochs=str(tfr.nave),
                    t_min=start_time,
                    t_max=end_time,
                    f_min=fmin,
                    f_max=fmax,
                    data_path=data_path,
                    cmap=NRcmap
                )